# 05. Сдвиг распределений

**Цель:** сравнить распределения признаков validation и test.
**Условия:** PSI ≥ 0,2 — сигнал для проверки, не доказательство ухудшения модели. Метки в расчёте не используются.

## Условия сравнения

Validation охватывает около суток, test — более восьми дней. Календарные признаки и накопленные счётчики могут сдвигаться без деградации модели.

[Итоговый отчёт](../reports/full/RESULTS.md)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Запустите notebook из корня проекта или notebooks")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from aml.pipeline import load_split, resolve_run, new_run
RUN_DIR = resolve_run(PROJECT_ROOT)
DATA_DIR = RUN_DIR / "data"
DRIFT_DIR = new_run(RUN_DIR, "drift")

In [2]:
reference, _ = load_split("val", DATA_DIR)
current, _ = load_split("test", DATA_DIR)
if list(reference.columns) != list(current.columns):
    raise ValueError("Feature schema changed")
rows = []
for col in reference.columns:
    a, b = reference[col], current[col]
    values = a.dropna().to_numpy()
    edges = np.unique(np.quantile(values, np.linspace(0, 1, 11))) if len(values) else np.array([])
    if len(edges) < 2:
        # Constant reference still detects a newly varying distribution.
        center = values[0] if len(values) else 0.0
        edges = np.array([-np.inf, np.nextafter(center, -np.inf), np.nextafter(center, np.inf), np.inf])
    else:
        edges = np.r_[-np.inf, edges, np.inf]
    p = np.r_[np.histogram(a.dropna(), edges)[0], a.isna().sum()].astype(float) + 0.5
    q = np.r_[np.histogram(b.dropna(), edges)[0], b.isna().sum()].astype(float) + 0.5
    p /= p.sum(); q /= q.sum()
    psi = float(np.sum((q-p)*np.log(q/p)))
    rows.append({"feature": col, "PSI": psi, "investigate": psi >= 0.2,
                 "missing_reference": a.isna().mean(), "missing_current": b.isna().mean()})
report = pd.DataFrame(rows).sort_values("PSI", ascending=False)
report.to_csv(DRIFT_DIR / "drift.csv", index=False)
report.to_html(DRIFT_DIR / "drift.html", index=False)
display(report.head(30))

,feature,PSI,investigate,missing_reference,missing_current
8,dow_sin,23.504297,True,0.0,0.0
4,day,4.931949,True,0.0,0.0
11,is_weekend,3.583104,True,0.0,0.0
3,dayofweek,3.466625,True,0.0,0.0
23,pair_prev_tx_log,1.823742,True,0.0,0.0
21,sender_prev_tx_log,0.402417,True,0.0,0.0
22,receiver_prev_tx_log,0.305225,True,0.0,0.0
28,sender_minutes_since_prev_log,0.218504,True,0.0,0.0
2,hour,0.097837,False,0.0,0.0
60,From_Bank__To_Bank__ote_200,0.090881,False,0.0,0.0


## Выводы

- PSI ≥ 0,2 у 8 из 65 признаков. Лидируют `dow_sin` (23,50), `day` (4,93) и `is_weekend` (3,58).
- Большая часть сигналов связана с календарём и накоплением истории; PSI денежных сумм — около 0,0017.
- Следующая проверка должна использовать сопоставимые временные окна и метрики качества после получения меток.